# 🧠 Emotion Detection — Advanced CNN Training on Google Colab

**GPU-Accelerated Training** | FER-2013 Dataset | 7 Emotions

Emotions: `Angry | Disgust | Fear | Happy | Sad | Surprise | Neutral`

---
### ⚙️ Setup
1. Go to **Runtime → Change runtime type → T4 GPU**
2. Run all cells top-to-bottom
3. Download `emotion_model.h5` from the Files panel at the end

In [ ]:
import tensorflow as tf
print('TensorFlow:', tf.__version__)
print('GPUs:', tf.config.list_physical_devices('GPU'))
print('GPU name:', tf.test.gpu_device_name() or 'No GPU')
!pip install -q kaggle visualkeras seaborn scikit-learn

In [ ]:
#  Download FER-2013 Dataset 
# OPTION A: Use kaggle API (recommended)
# Upload your kaggle.json API token to Colab first:
#   Files panel → upload → kaggle.json
# Then run:

import os

if os.path.exists('/content/kaggle.json'):
    !mkdir -p ~/.kaggle
    !cp /content/kaggle.json ~/.kaggle/
    !chmod 600 ~/.kaggle/kaggle.json
    !kaggle datasets download -d msambare/fer2013 --unzip -p /content/dataset
    print('✅ Dataset downloaded via Kaggle API')
else:
    print('⚠️  kaggle.json not found.')
    print('  OPTION B: Mount Google Drive and set DATASET_PATH below')
    print('  OR manually upload dataset folder and set paths below')

# OPTION B: Mount Google Drive
# from google.colab import drive
# drive.mount('/content/drive')
# DATASET_PATH = '/content/drive/MyDrive/fer2013/'

# After download, folder structure should be:
# /content/dataset/train/Angry/, /Happy/, ... (7 folders)
# /content/dataset/test/Angry/,  /Happy/, ...

In [ ]:
import pathlib

TRAIN_DIR = '/content/dataset/train'
TEST_DIR  = '/content/dataset/test'

for split, d in [('train', TRAIN_DIR), ('test', TEST_DIR)]:
    counts = {p.name: len(list(p.glob('*'))) for p in pathlib.Path(d).iterdir() if p.is_dir()}
    total = sum(counts.values())
    print(f'\n{split.upper()} ({total} images):')
    for emotion, n in sorted(counts.items()):
        bar = '█' * (n // 100)
        print(f'  {emotion:<12} {n:>5}  {bar}')

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import random, glob

LABELS = ['Angry','Disgust','Fear','Happy','Sad','Surprise','Neutral']
fig, axes = plt.subplots(2, 7, figsize=(18, 6))
fig.suptitle('Sample Training Images per Emotion', fontsize=16, fontweight='bold')

for col, emotion in enumerate(LABELS):
    imgs = glob.glob(f'{TRAIN_DIR}/{emotion}/*.jpg') + \
           glob.glob(f'{TRAIN_DIR}/{emotion}/*.png')
    for row in range(2):
        if imgs:
            img = mpimg.imread(random.choice(imgs))
            axes[row, col].imshow(img, cmap='gray')
        axes[row, col].axis('off')
        if row == 0:
            axes[row, col].set_title(emotion, fontsize=10, fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import (
    EarlyStopping, ReduceLROnPlateau, ModelCheckpoint, TensorBoard
)
import datetime

IMG_SIZE   = 48
BATCH_SIZE = 64
EPOCHS     = 100
NUM_CLASSES = 7
SEED = 42
tf.random.set_seed(SEED)
np.random.seed(SEED)

print('All imports OK ✅')

In [ ]:
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=30,
    width_shift_range=0.15,
    height_shift_range=0.15,
    zoom_range=0.2,
    horizontal_flip=True,
    brightness_range=[0.7, 1.3],
    shear_range=0.1,
    fill_mode='nearest'
)

test_datagen = ImageDataGenerator(rescale=1./255)

train_gen = train_datagen.flow_from_directory(
    TRAIN_DIR,
    target_size=(IMG_SIZE, IMG_SIZE),
    color_mode='grayscale',
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=True,
    seed=SEED
)

test_gen = test_datagen.flow_from_directory(
    TEST_DIR,
    target_size=(IMG_SIZE, IMG_SIZE),
    color_mode='grayscale',
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False
)

print('Class indices:', train_gen.class_indices)

In [ ]:
def build_emotion_model(num_classes=7):
    inputs = keras.Input(shape=(IMG_SIZE, IMG_SIZE, 1))

    # ── Block 1
    x = layers.Conv2D(64, 3, padding='same', use_bias=False)(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.Conv2D(64, 3, padding='same', use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.MaxPooling2D(2)(x)
    x = layers.Dropout(0.25)(x)

    # ── Block 2
    x = layers.Conv2D(128, 3, padding='same', use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.Conv2D(128, 3, padding='same', use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.MaxPooling2D(2)(x)
    x = layers.Dropout(0.25)(x)

    # ── Block 3
    x = layers.Conv2D(256, 3, padding='same', use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.Conv2D(256, 3, padding='same', use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.MaxPooling2D(2)(x)
    x = layers.Dropout(0.25)(x)

    # ── Block 4 (extra depth)
    x = layers.Conv2D(512, 3, padding='same', use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.MaxPooling2D(2)(x)
    x = layers.Dropout(0.25)(x)

    # ── Classifier head
    x = layers.GlobalAveragePooling2D()(x)   # better than Flatten
    x = layers.Dense(512, use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.Dropout(0.5)(x)
    x = layers.Dense(256, use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(num_classes, activation='softmax')(x)

    return keras.Model(inputs, outputs, name='EmotionCNN_v2')


model = build_emotion_model()
model.summary()
print(f'\nTotal parameters: {model.count_params():,}')

In [ ]:
from sklearn.utils.class_weight import compute_class_weight

class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.arange(NUM_CLASSES),
    y=train_gen.classes
)
class_weight_dict = dict(enumerate(class_weights))
print('Class weights:', {LABELS[k]: round(v,3) for k,v in class_weight_dict.items()})

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss='categorical_crossentropy',
    metrics=['accuracy', keras.metrics.TopKCategoricalAccuracy(k=2, name='top2_acc')]
)

In [ ]:
import os
os.makedirs('/content/model', exist_ok=True)

log_dir = '/content/logs/' + datetime.datetime.now().strftime('%Y%m%d-%H%M%S')

callbacks = [
    ModelCheckpoint(
        '/content/model/emotion_model.h5',
        monitor='val_accuracy',
        save_best_only=True,
        verbose=1
    ),
    EarlyStopping(
        monitor='val_accuracy',
        patience=15,
        restore_best_weights=True,
        verbose=1
    ),
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=5,
        min_lr=1e-6,
        verbose=1
    ),
    TensorBoard(log_dir=log_dir, histogram_freq=1)
]

print('Callbacks configured ✅')

In [ ]:
print('🚀 Starting training...')
history = model.fit(
    train_gen,
    validation_data=test_gen,
    epochs=EPOCHS,
    callbacks=callbacks,
    class_weight=class_weight_dict,
    verbose=1
)
print('\n✅ Training complete!')

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
fig.suptitle('Training History', fontsize=16, fontweight='bold')

# Accuracy
axes[0].plot(history.history['accuracy'], label='Train', linewidth=2)
axes[0].plot(history.history['val_accuracy'], label='Validation', linewidth=2)
axes[0].set_title('Accuracy')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Loss
axes[1].plot(history.history['loss'], label='Train', linewidth=2)
axes[1].plot(history.history['val_loss'], label='Validation', linewidth=2)
axes[1].set_title('Loss')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('/content/model/training_curves.png', dpi=150, bbox_inches='tight')
plt.show()

best_val_acc = max(history.history['val_accuracy'])
print(f'\n🏆 Best Validation Accuracy: {best_val_acc:.4f} ({best_val_acc*100:.2f}%)')

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns

# Predictions on test set
y_pred_probs = model.predict(test_gen, verbose=1)
y_pred = np.argmax(y_pred_probs, axis=1)
y_true = test_gen.classes

# Confusion matrix
cm = confusion_matrix(y_true, y_pred)
cm_pct = cm.astype('float') / cm.sum(axis=1, keepdims=True)  # row-normalize

plt.figure(figsize=(10, 8))
sns.heatmap(
    cm_pct, annot=True, fmt='.2f', cmap='Blues',
    xticklabels=LABELS, yticklabels=LABELS,
    linewidths=0.5, linecolor='white'
)
plt.title('Confusion Matrix (row-normalized)', fontsize=14, fontweight='bold')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.tight_layout()
plt.savefig('/content/model/confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

# Classification report
print('\nClassification Report:')
print(classification_report(y_true, y_pred, target_names=LABELS))